# Notebook 6: Classification Metrics — Accuracy, Precision, Recall, F1 & F-Beta

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 3 hours  
**Week 7 — Model Evaluation, Validation & Metrics**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Build and interpret a confusion matrix and explain each cell's business meaning
2. Compute accuracy, precision, recall, F1, F-beta, and specificity from scratch
3. Explain why accuracy is misleading for imbalanced fraud data
4. Choose an appropriate threshold for a given business objective
5. Select the right metric for a given cost asymmetry (FP cost vs FN cost)

---

## Why Does This Matter?

**Metrics are the lens through which you see your model.** Choose the wrong lens and you are blind to the model's real failures.

A model that predicts "legit" for every single transaction achieves **98% accuracy** on a dataset where fraud is 2% of transactions. This model is completely useless — it catches zero fraud — yet the accuracy score tells you nothing is wrong. This is why fraud detection teams never use accuracy as their primary metric.

Every metric corresponds to a business question:
- **"How trustworthy are our fraud alerts?"** → Precision
- **"How much fraud are we actually catching?"** → Recall
- **"Do we care equally about false alarms and missed fraud?"** → F1
- **"Missing fraud costs us 5x more than a false alarm"** → F-beta (β=2)

## Real-World Analogy: The Security Guard at a Bank Branch

Imagine a bank employs a security guard who must decide whether to stop each customer entering the branch.

- **True Positive (TP):** Guard stops someone — and they really are a thief. Caught a fraudster.
- **False Positive (FP):** Guard stops someone — but they are an innocent customer. Embarrassing and costly.
- **True Negative (TN):** Guard lets someone through — and they really are a legitimate customer. All good.
- **False Negative (FN):** Guard lets someone through — and they are a thief. Money is stolen. Worst outcome.

**Accuracy** = (all correct decisions) / (all decisions) — but if 999 out of 1000 customers are honest, a guard who never stops anyone gets 99.9% accuracy while being completely useless.

**Precision** = of everyone the guard stops, what fraction was actually a thief? Low precision = guard stops too many innocent people → customer complaints.

**Recall** = of all actual thieves, what fraction did the guard catch? Low recall = too many thieves slip through → financial loss.

**F1** = a single number that penalises being very bad at either.

**F-beta** = adjusts the balance. The bank decides: "A missed thief costs us £1,000. A wrongly-stopped customer costs us £200 in service recovery." β encodes that ratio.

---

## The Confusion Matrix

```
                    Predicted LEGIT    Predicted FRAUD
Actual LEGIT    |       TN            |      FP          |  ← False alarm (blocked innocent customer)
Actual FRAUD    |       FN            |      TP          |  ← Missed fraud  /  Caught fraud
```

Remember: **rows** = actual, **columns** = predicted. The main diagonal = correct predictions.

In the sklearn convention, the positive class is labelled 1 (fraud). The matrix is:

```
sklearn confusion_matrix([[TN, FP],
                           [FN, TP]])
```

In [ ]:
# ── Cell 3: Imports and configuration ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    accuracy_score, precision_recall_curve, fbeta_score,
    roc_curve, balanced_accuracy_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

print("Libraries loaded.")

In [ ]:
# ── Cell 4: Generate fraud dataset (n=5000, 3% fraud) and train model ─────────
# Realistic imbalance: 3% fraud is typical for credit card transactions

np.random.seed(42)
N        = 5000
n_fraud  = int(N * 0.03)       # 150 fraud cases
n_legit  = N - n_fraud          # 4850 legit cases

# Legit transactions
legit = pd.DataFrame({
    'transaction_amount': np.random.lognormal(4.0, 0.8, n_legit),
    'time_of_day':        np.random.uniform(6, 22, n_legit),
    'merchant_risk':      np.random.beta(2, 8, n_legit),
    'txn_today':          np.random.poisson(3, n_legit).astype(float),
    'dist_from_home':     np.random.exponential(15, n_legit),
    'velocity_score':     np.random.normal(0.2, 0.1, n_legit).clip(0, 1),
    'is_fraud': 0
})

# Fraud: larger amounts, late-night, risky merchants, high velocity
fraud = pd.DataFrame({
    'transaction_amount': np.random.lognormal(5.5, 1.0, n_fraud),
    'time_of_day':        np.random.choice(
                              np.concatenate([np.arange(0, 5), np.arange(22, 25)]),
                              n_fraud).astype(float) % 24,
    'merchant_risk':      np.random.beta(8, 2, n_fraud),
    'txn_today':          np.random.poisson(9, n_fraud).astype(float),
    'dist_from_home':     np.random.exponential(80, n_fraud),
    'velocity_score':     np.random.normal(0.8, 0.1, n_fraud).clip(0, 1),
    'is_fraud': 1
})

df = pd.concat([legit, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)
feature_cols = [c for c in df.columns if c != 'is_fraud']
X, y = df[feature_cols].values, df['is_fraud'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Train logistic regression
pipe = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', LogisticRegression(C=0.5, max_iter=500, random_state=42))
])
pipe.fit(X_train, y_train)

y_pred_prob = pipe.predict_proba(X_test)[:, 1]   # fraud probability scores
y_pred_05   = (y_pred_prob >= 0.5).astype(int)   # default threshold = 0.5

print(f"Training set: {X_train.shape[0]} samples ({y_train.sum()} fraud)")
print(f"Test set:     {X_test.shape[0]} samples ({y_test.sum()} fraud = {y_test.mean()*100:.1f}%)")

In [ ]:
# ── Cell 5: Manual confusion matrix computation ────────────────────────────────
# Understanding from first principles — before using sklearn

def manual_confusion_matrix(y_true, y_pred, positive_class=1):
    """
    Returns (TP, FP, TN, FN) for binary classification.
    Positive class = fraud (label 1).
    """
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))   # predicted fraud, IS fraud
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))   # predicted fraud, NOT fraud
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))   # predicted legit, IS legit
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))   # predicted legit, IS fraud
    return TP, FP, TN, FN


TP, FP, TN, FN = manual_confusion_matrix(y_test, y_pred_05)

print("Manual Confusion Matrix Components (threshold = 0.50):")
print(f"  TP = {TP:4d}  | Predicted FRAUD, actually FRAUD → Caught fraudster")
print(f"  FP = {FP:4d}  | Predicted FRAUD, actually LEGIT → Blocked innocent customer  ← customer pain")
print(f"  TN = {TN:4d}  | Predicted LEGIT, actually LEGIT → Correct pass-through")
print(f"  FN = {FN:4d}  | Predicted LEGIT, actually FRAUD → Missed fraudster           ← financial loss")
print(f"\n  Total predictions: {TP+FP+TN+FN}")
print(f"  Fraud in test set: {TP+FN}  ({(TP+FN)/(TP+FP+TN+FN)*100:.1f}%)")

In [ ]:
# ── Cell 6: Seaborn heatmap — confusion matrix with full annotation ───────────

# Build 2x2 matrix
cm       = np.array([[TN, FP], [FN, TP]])
cm_norm  = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels_count = [
    [f'TN = {TN}\n(Correct LEGIT)', f'FP = {FP}\n← False Alarm\n(Blocked innocent)'],
    [f'FN = {FN}\n← MISSED FRAUD\n(Most costly!)',  f'TP = {TP}\n(Caught fraud)']
]

# ---- Left: raw counts ----
ax = axes[0]
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=['Predicted LEGIT', 'Predicted FRAUD'],
            yticklabels=['Actual LEGIT', 'Actual FRAUD'], ax=ax,
            linewidths=2, linecolor='white', cbar_kws={'label': 'Count'})
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j + 0.5, i + 0.5, labels_count[i][j],
                ha='center', va='center', fontsize=9,
                color=color, fontweight='bold')
ax.set_title('Confusion Matrix — Raw Counts\n(threshold = 0.50)', fontsize=11)

# ---- Right: row-normalised (rates) ----
ax2 = axes[1]
labels_norm = [
    [f'TNR = {cm_norm[0,0]:.1%}\n(Specificity)', f'FPR = {cm_norm[0,1]:.1%}\n(Fall-out)'],
    [f'FNR = {cm_norm[1,0]:.1%}\n(Miss Rate)',    f'TPR = {cm_norm[1,1]:.1%}\n(Recall/Sensitivity)']
]
sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Oranges',
            xticklabels=['Predicted LEGIT', 'Predicted FRAUD'],
            yticklabels=['Actual LEGIT', 'Actual FRAUD'], ax=ax2,
            linewidths=2, linecolor='white', cbar_kws={'label': 'Rate (per row)'})
for i in range(2):
    for j in range(2):
        color = 'white' if cm_norm[i, j] > 0.6 else 'black'
        ax2.text(j + 0.5, i + 0.5, labels_norm[i][j],
                 ha='center', va='center', fontsize=9,
                 color=color, fontweight='bold')
ax2.set_title('Confusion Matrix — Row-Normalised Rates\n(each row sums to 100%)', fontsize=11)

plt.suptitle('Fraud Detection — Confusion Matrix Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: All metrics from scratch ─────────────────────────────────────────
# Derive every metric from the four confusion matrix cells: TP, FP, TN, FN

def compute_all_metrics(TP, FP, TN, FN, beta=1.0):
    """Compute classification metrics from raw confusion matrix values."""
    total   = TP + FP + TN + FN
    # Safety: avoid division by zero
    eps = 1e-9

    accuracy          = (TP + TN) / total
    precision         = TP / (TP + FP + eps)        # of predicted fraud, what % is real?
    recall            = TP / (TP + FN + eps)        # of actual fraud, what % caught?
    specificity       = TN / (TN + FP + eps)        # of actual legit, what % passed?
    f1                = 2 * precision * recall / (precision + recall + eps)
    f_beta            = (1 + beta**2) * precision * recall / (
                          beta**2 * precision + recall + eps)
    balanced_acc      = (recall + specificity) / 2
    false_pos_rate    = FP / (FP + TN + eps)        # = 1 - specificity
    false_neg_rate    = FN / (FN + TP + eps)        # = 1 - recall

    return {
        'Accuracy':          accuracy,
        'Precision':         precision,
        'Recall (TPR)':      recall,
        'Specificity (TNR)': specificity,
        'False Positive Rate': false_pos_rate,
        'False Negative Rate': false_neg_rate,
        'F1 Score':          f1,
        f'F{beta} Score':    f_beta,
        'Balanced Accuracy': balanced_acc,
    }

metrics_manual = compute_all_metrics(TP, FP, TN, FN, beta=1.0)

print("Metrics computed from scratch (threshold = 0.50):")
print("-" * 55)
for name, val in metrics_manual.items():
    print(f"  {name:<25} = {val:.4f}  ({val*100:.1f}%)")

In [ ]:
# ── Cell 8: Verify against sklearn metrics ────────────────────────────────────

print("Verifying against sklearn.metrics (threshold = 0.50):")
print("=" * 55)

sk_accuracy  = accuracy_score(y_test, y_pred_05)
sk_precision = precision_score(y_test, y_pred_05, zero_division=0)
sk_recall    = recall_score(y_test, y_pred_05, zero_division=0)
sk_f1        = f1_score(y_test, y_pred_05, zero_division=0)
sk_bal_acc   = balanced_accuracy_score(y_test, y_pred_05)

verify_pairs = [
    ('Accuracy',         metrics_manual['Accuracy'],        sk_accuracy),
    ('Precision',        metrics_manual['Precision'],       sk_precision),
    ('Recall',           metrics_manual['Recall (TPR)'],    sk_recall),
    ('F1',               metrics_manual['F1 Score'],        sk_f1),
    ('Balanced Accuracy',metrics_manual['Balanced Accuracy'], sk_bal_acc),
]

for name, manual_val, sklearn_val in verify_pairs:
    match = abs(manual_val - sklearn_val) < 1e-6
    status = 'MATCH' if match else 'MISMATCH!'
    print(f"  {name:<20} | Manual={manual_val:.5f} | sklearn={sklearn_val:.5f} | {status}")

print("\nDemonstrating the accuracy trap:")
y_always_legit = np.zeros_like(y_test)   # dumb model: predict LEGIT for everything
print(f"  'Always Legit' model accuracy: {accuracy_score(y_test, y_always_legit):.4f}")
print(f"  'Always Legit' recall:         {recall_score(y_test, y_always_legit, zero_division=0):.4f}")
print(f"  → 97%+ accuracy, zero fraud caught. Accuracy is useless here!")

## Understanding Each Metric

### Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + FP + TN + FN}$$

**What it measures:** Of ALL predictions (fraud and legit), what fraction was correct?  
**Fatal flaw for fraud:** A dataset with 97% legit transactions gives 97% accuracy to a model that always predicts "legit" — and catches zero fraud.  
**When it is useful:** Balanced classes AND equal cost of FP and FN.

---

### Precision
$$\text{Precision} = \frac{TP}{TP + FP}$$

**What it measures:** Of all transactions we FLAGGED as fraud, what fraction really was fraud?  
**Business meaning:** Precision = trustworthiness of fraud alerts. Low precision = customer service nightmare (too many blocked innocent customers).  
**How to improve:** Raise the classification threshold → fewer predictions → fewer FPs.

---

### Recall (Sensitivity, True Positive Rate)
$$\text{Recall} = \frac{TP}{TP + FN}$$

**What it measures:** Of all ACTUAL fraud transactions, what fraction did we catch?  
**Business meaning:** Recall = how much fraud we stop. Low recall = fraudsters slip through → financial loss.  
**How to improve:** Lower the classification threshold → flag more transactions → fewer FNs.

---

### The Precision-Recall Tradeoff

Any change in threshold that increases precision MUST decrease recall (and vice versa):
- **Higher threshold:** the model only flags high-confidence fraud → fewer FPs (good precision), but misses borderline cases → more FNs (bad recall)
- **Lower threshold:** model flags everything suspicious → catches more fraud (good recall), but also catches more innocent customers (bad precision)

This is not a model failure — it is a **business decision:** what is the relative cost of FP vs FN?

---

### F1 Score
$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

F1 is the **harmonic mean** of precision and recall. The harmonic mean is always lower than the arithmetic mean when the two values differ — it PENALISES imbalance between precision and recall. A model with Precision=1.0 and Recall=0.01 gets F1 = 0.02, not 0.5.

**Use F1 when:** FP and FN have roughly equal costs, and you want a single metric.

---

### F-Beta Score
$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

- **β > 1:** recall matters more (e.g., β=2: recall is twice as important as precision). Use for: cancer screening (missing a case is catastrophic), fraud where FN >> FP cost.
- **β < 1:** precision matters more (e.g., β=0.5: precision twice as important). Use for: spam filter (false alarms are very annoying), or when chasing fraud alerts is expensive.

In [ ]:
# ── Cell 10: Threshold sweep — precision, recall, F0.5, F1, F2 vs threshold ───
# Shows how every metric changes as we slide the decision boundary

thresholds = np.linspace(0.01, 0.99, 200)

results = {
    'threshold': thresholds,
    'precision': [], 'recall': [],
    'f0.5': [], 'f1': [], 'f2': []
}

for thr in thresholds:
    y_pred_t = (y_pred_prob >= thr).astype(int)
    TP_t = int(np.sum((y_test == 1) & (y_pred_t == 1)))
    FP_t = int(np.sum((y_test == 0) & (y_pred_t == 1)))
    FN_t = int(np.sum((y_test == 1) & (y_pred_t == 0)))
    eps  = 1e-9

    prec = TP_t / (TP_t + FP_t + eps)
    rec  = TP_t / (TP_t + FN_t + eps)

    results['precision'].append(prec)
    results['recall'].append(rec)
    results['f0.5'].append((1 + 0.5**2) * prec * rec / (0.5**2 * prec + rec + eps))
    results['f1'].append(2 * prec * rec / (prec + rec + eps))
    results['f2'].append((1 + 2**2) * prec * rec / (2**2 * prec + rec + eps))

res_df = pd.DataFrame(results)

# Find optimal threshold for each F-score
opt_f05 = thresholds[np.argmax(results['f0.5'])]
opt_f1  = thresholds[np.argmax(results['f1'])]
opt_f2  = thresholds[np.argmax(results['f2'])]

print("Optimal thresholds:")
print(f"  F0.5 (precision-focused): threshold = {opt_f05:.3f} | max F0.5 = {max(results['f0.5']):.4f}")
print(f"  F1   (balanced):          threshold = {opt_f1:.3f}  | max F1   = {max(results['f1']):.4f}")
print(f"  F2   (recall-focused):    threshold = {opt_f2:.3f}  | max F2   = {max(results['f2']):.4f}")
print(f"\nNote: F2 has a LOWER optimal threshold (flags more = higher recall).")
print(f"      F0.5 has a HIGHER optimal threshold (flags less = higher precision).")

In [ ]:
# ── Cell 11: Threshold sweep visualisation ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ---- Left: all metrics vs threshold ----
ax = axes[0]
ax.plot(thresholds, results['precision'], color='steelblue',  label='Precision',      lw=2)
ax.plot(thresholds, results['recall'],    color='tomato',     label='Recall',         lw=2)
ax.plot(thresholds, results['f0.5'],      color='purple',     label='F0.5 (precision focus)', lw=1.5, linestyle='-.')
ax.plot(thresholds, results['f1'],        color='seagreen',   label='F1  (balanced)', lw=2)
ax.plot(thresholds, results['f2'],        color='darkorange', label='F2  (recall focus)', lw=1.5, linestyle='--')

# Mark optimal thresholds
ax.axvline(opt_f05, color='purple',     linestyle=':', alpha=0.7, label=f'Best F0.5 thr={opt_f05:.2f}')
ax.axvline(opt_f1,  color='seagreen',   linestyle=':', alpha=0.7, label=f'Best F1   thr={opt_f1:.2f}')
ax.axvline(opt_f2,  color='darkorange', linestyle=':', alpha=0.7, label=f'Best F2   thr={opt_f2:.2f}')
ax.axvline(0.5, color='black', linestyle='--', alpha=0.3, label='Default thr=0.5')

ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Metric Value')
ax.set_title('All Metrics vs Classification Threshold\nFraud Detection (3% fraud rate)', fontsize=11)
ax.legend(fontsize=8, loc='center left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

# ---- Right: precision-recall curve ----
ax2 = axes[1]
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_test, y_pred_prob)

# Colour by threshold
sc = ax2.scatter(rec_curve[:-1], prec_curve[:-1],
                 c=thr_curve, cmap='RdYlGn_r', s=15, alpha=0.8)
plt.colorbar(sc, ax=ax2, label='Threshold')

# Mark key operating points
for label, thr_val, col in [('F0.5', opt_f05, 'purple'),
                              ('F1',   opt_f1,  'seagreen'),
                              ('F2',   opt_f2,  'darkorange')]:
    idx = np.argmin(np.abs(thr_curve - thr_val))
    ax2.scatter(rec_curve[idx], prec_curve[idx],
                color=col, s=150, zorder=5, edgecolors='black',
                label=f'Opt {label} ({rec_curve[idx]:.2f}, {prec_curve[idx]:.2f})')

ax2.set_xlabel('Recall (Fraud Catch Rate)')
ax2.set_ylabel('Precision (Alert Trustworthiness)')
ax2.set_title('Precision-Recall Curve\nColour = threshold value', fontsize=11)
ax2.legend(fontsize=8)

plt.suptitle('Threshold Matters: Different Metrics Prefer Different Operating Points',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: F-beta comparison — show β controls the precision-recall tradeoff ─

beta_values = [0.25, 0.5, 1.0, 2.0, 3.0]
beta_results = {}

for beta in beta_values:
    f_scores = []
    for thr in thresholds:
        y_pred_t = (y_pred_prob >= thr).astype(int)
        TP_t = int(np.sum((y_test == 1) & (y_pred_t == 1)))
        FP_t = int(np.sum((y_test == 0) & (y_pred_t == 1)))
        FN_t = int(np.sum((y_test == 1) & (y_pred_t == 0)))
        eps  = 1e-9
        prec = TP_t / (TP_t + FP_t + eps)
        rec  = TP_t / (TP_t + FN_t + eps)
        fb   = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + eps)
        f_scores.append(fb)
    opt_thr = thresholds[np.argmax(f_scores)]
    opt_val = max(f_scores)
    y_pred_opt = (y_pred_prob >= opt_thr).astype(int)
    TP_o = int(np.sum((y_test == 1) & (y_pred_opt == 1)))
    FP_o = int(np.sum((y_test == 0) & (y_pred_opt == 1)))
    FN_o = int(np.sum((y_test == 1) & (y_pred_opt == 0)))
    eps  = 1e-9
    beta_results[beta] = {
        'opt_threshold': opt_thr,
        'max_fbeta':     opt_val,
        'precision':     TP_o / (TP_o + FP_o + eps),
        'recall':        TP_o / (TP_o + FN_o + eps),
        'TP': TP_o, 'FP': FP_o, 'FN': FN_o
    }

print(f"{'β':>6} | {'Opt Thr':>9} | {'Max Fβ':>8} | {'Precision':>10} | {'Recall':>8} | {'TP':>4} | {'FP':>4} | {'FN':>4}")
print("-" * 70)
for beta, r in beta_results.items():
    focus = '← recall focus' if beta >= 2 else ('← precision focus' if beta <= 0.5 else '← balanced')
    print(f"  {beta:>4} | {r['opt_threshold']:>9.3f} | {r['max_fbeta']:>8.4f} | "
          f"{r['precision']:>10.4f} | {r['recall']:>8.4f} | {r['TP']:>4} | {r['FP']:>4} | {r['FN']:>4}  {focus}")

In [ ]:
# ── Cell 13: F-beta visualisation ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ---- Left: F-beta curves on P-R space ----
ax = axes[0]
prec_grid = np.linspace(0.01, 1.0, 300)
palette   = plt.cm.plasma(np.linspace(0.1, 0.9, len(beta_values)))

for beta, color in zip(beta_values, palette):
    r_vals = [beta_results[beta]['recall']]
    p_vals = [beta_results[beta]['precision']]
    ax.scatter(r_vals, p_vals, color=color, s=200, zorder=5,
               edgecolors='black', linewidth=1,
               label=f'β={beta} | thr={beta_results[beta]["opt_threshold"]:.2f}')

# Plot precision-recall curve as background
ax.plot(rec_curve, prec_curve, 'grey', alpha=0.4, lw=2, label='PR Curve')

# Annotate direction of β
ax.annotate('↑ β → lower threshold\n  → higher recall',
            xy=(0.65, 0.55), fontsize=9, color='darkred',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.annotate('↓ β → higher threshold\n  → higher precision',
            xy=(0.2, 0.8), fontsize=9, color='navy',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.8))

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Optimal Operating Points for Each F-beta\nHigher β → lower threshold → recall focus', fontsize=10)
ax.legend(fontsize=8, loc='lower left')

# ---- Right: TP / FP / FN bars for each beta ----
ax2 = axes[1]
x    = np.arange(len(beta_values))
tps  = [beta_results[b]['TP'] for b in beta_values]
fps  = [beta_results[b]['FP'] for b in beta_values]
fns  = [beta_results[b]['FN'] for b in beta_values]

w = 0.25
ax2.bar(x - w, tps, width=w, label='TP (caught fraud)',     color='seagreen',  alpha=0.8)
ax2.bar(x,     fps, width=w, label='FP (false alarm)',      color='steelblue', alpha=0.8)
ax2.bar(x + w, fns, width=w, label='FN (missed fraud)',     color='tomato',    alpha=0.8)

ax2.set_xticks(x)
ax2.set_xticklabels([f'β={b}' for b in beta_values])
ax2.set_ylabel('Count (test set)')
ax2.set_title('Business Trade-off at Each β Optimal Threshold\nMore recall → more TP but also more FP', fontsize=10)
ax2.legend(fontsize=9)

plt.suptitle('F-Beta Score: Encoding Business Priorities into Evaluation',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14: Specificity and Balanced Accuracy ────────────────────────────────
# Specificity is the recall for the NEGATIVE class

def specificity_at_threshold(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    TN_t = np.sum((y_true == 0) & (y_pred == 0))
    FP_t = np.sum((y_true == 0) & (y_pred == 1))
    return TN_t / (TN_t + FP_t + 1e-9)

specificities = [specificity_at_threshold(y_test, y_pred_prob, t) for t in thresholds]
recalls_curve = results['recall']
bal_accs      = [(r + s) / 2 for r, s in zip(recalls_curve, specificities)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, recalls_curve,  label='Recall (Sensitivity = TPR)', color='tomato', lw=2)
ax.plot(thresholds, specificities,  label='Specificity (TNR)',           color='steelblue', lw=2)
ax.plot(thresholds, bal_accs,       label='Balanced Accuracy = (TPR+TNR)/2',
        color='purple', lw=2, linestyle='--')
ax.plot(thresholds, results['f1'],  label='F1', color='seagreen', lw=1.5, linestyle='-.')

# Best balanced accuracy
opt_bal = thresholds[np.argmax(bal_accs)]
ax.axvline(opt_bal, color='purple', linestyle=':', alpha=0.8,
           label=f'Best Balanced Acc threshold = {opt_bal:.2f}')

ax.set_xlabel('Threshold')
ax.set_ylabel('Metric Value')
ax.set_title('Specificity, Recall, Balanced Accuracy vs Threshold\n'
             'Balanced Acc = arithmetic mean of TPR and TNR — good for imbalanced data', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print(f"At threshold = {opt_bal:.3f}:")
spec_opt = specificities[np.argmax(bal_accs)]
rec_opt  = recalls_curve[np.argmax(bal_accs)]
print(f"  Recall:            {rec_opt:.4f}")
print(f"  Specificity:       {spec_opt:.4f}")
print(f"  Balanced Accuracy: {(rec_opt + spec_opt)/2:.4f}")

In [ ]:
# ── Cell 15: sklearn classification_report — full per-class breakdown ──────────
# Production-ready summary: per-class and averaged metrics

print("Classification Report at default threshold (0.50):")
print("=" * 60)
print(classification_report(
    y_test, y_pred_05,
    target_names=['Legit (0)', 'Fraud (1)'],
    digits=4
))

# Show report at optimal F2 threshold (recall-focused)
y_pred_f2 = (y_pred_prob >= opt_f2).astype(int)
print(f"\nClassification Report at F2-optimal threshold ({opt_f2:.3f}):")
print("=" * 60)
print(classification_report(
    y_test, y_pred_f2,
    target_names=['Legit (0)', 'Fraud (1)'],
    digits=4
))

In [ ]:
# ── Cell 16: Business scenario comparison ─────────────────────────────────────
# Scenario A: "We cannot miss more than 10% of fraud" → maximise recall ≥ 0.90
# Scenario B: "Fewer than 1% of alerts can be false alarms" → maximise precision ≥ 0.99

# --- Scenario A: recall-constrained (minimise FN) ---
target_recall  = 0.90
# Find lowest threshold that achieves ≥ 90% recall
valid_thrs_A = [thr for thr, rec in zip(thresholds, results['recall']) if rec >= target_recall]
best_thr_A   = max(valid_thrs_A) if valid_thrs_A else 0.01  # highest thr that still gets recall
y_pred_A = (y_pred_prob >= best_thr_A).astype(int)
TP_A, FP_A, TN_A, FN_A = manual_confusion_matrix(y_test, y_pred_A)

# --- Scenario B: precision-constrained (minimise FP) ---
target_prec  = 0.90
# Find lowest threshold that achieves ≥ 90% precision (highest recall while maintaining precision)
valid_thrs_B = [(thr, rec) for thr, prec, rec in
                zip(thresholds, results['precision'], results['recall'])
                if prec >= target_prec]
best_thr_B, _ = min(valid_thrs_B, key=lambda x: x[0]) if valid_thrs_B else (0.99, 0)
y_pred_B = (y_pred_prob >= best_thr_B).astype(int)
TP_B, FP_B, TN_B, FN_B = manual_confusion_matrix(y_test, y_pred_B)

print("Business Scenario Comparison")
print("=" * 70)
print(f"{'Metric':<25} | {'Default (0.50)':>15} | {'Scenario A':>15} | {'Scenario B':>15}")
print(f"{'':25} | {'':15} | {'Recall ≥ 90%':>15} | {'Precision ≥ 90%':>15}")
print("-" * 70)

for label, tp, fp, tn, fn in [
    ('Default', TP, FP, TN, FN),
    ('A', TP_A, FP_A, TN_A, FN_A),
    ('B', TP_B, FP_B, TN_B, FN_B)
]:
    pass

rows = [
    ('Threshold',   [0.50,                     best_thr_A,                best_thr_B]),
    ('TP (caught)', [TP,                        TP_A,                      TP_B]),
    ('FP (alarm)',  [FP,                        FP_A,                      FP_B]),
    ('FN (missed)', [FN,                        FN_A,                      FN_B]),
    ('Precision',   [TP/(TP+FP+1e-9),          TP_A/(TP_A+FP_A+1e-9),    TP_B/(TP_B+FP_B+1e-9)]),
    ('Recall',      [TP/(TP+FN+1e-9),          TP_A/(TP_A+FN_A+1e-9),    TP_B/(TP_B+FN_B+1e-9)]),
    ('F1',          [2*TP/(2*TP+FP+FN+1e-9),  2*TP_A/(2*TP_A+FP_A+FN_A+1e-9), 2*TP_B/(2*TP_B+FP_B+FN_B+1e-9)]),
]
for name, vals in rows:
    fmt = '.4f' if isinstance(vals[0], float) else 'd'
    print(f"  {name:<23} | {vals[0]:>15{fmt}} | {vals[1]:>15{fmt}} | {vals[2]:>15{fmt}}")

print("\nScenario A trades precision for recall (more false alarms, fewer missed fraud).")
print("Scenario B trades recall for precision (more missed fraud, fewer false alarms).")

In [ ]:
# ── Cell 17: Business cost comparison visualisation ───────────────────────────
# Quantify the actual financial impact of each operating point

# Assumed costs (USD)
COST_FN = 500    # missing a fraud case → avg loss per fraud transaction
COST_FP = 30     # false alarm → cost of investigation + customer friction

scenarios = {
    'Default (thr=0.50)': {'TP': TP,   'FP': FP,   'FN': FN},
    'Scenario A (recall≥90%)': {'TP': TP_A, 'FP': FP_A, 'FN': FN_A},
    'Scenario B (precision≥90%)': {'TP': TP_B, 'FP': FP_B, 'FN': FN_B},
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ---- Left: confusion matrices side-by-side ----
ax = axes[0]
x   = np.arange(3)
w   = 0.2
tps = [s['TP'] for s in scenarios.values()]
fps = [s['FP'] for s in scenarios.values()]
fns = [s['FN'] for s in scenarios.values()]

ax.bar(x - w, tps, width=w, label='TP (caught fraud)', color='seagreen', alpha=0.85)
ax.bar(x,     fps, width=w, label='FP (false alarm)',  color='steelblue', alpha=0.85)
ax.bar(x + w, fns, width=w, label='FN (missed fraud)', color='tomato',   alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(scenarios.keys(), rotation=10, ha='right', fontsize=9)
ax.set_ylabel('Count')
ax.set_title('Confusion Matrix Counts\nfor Each Business Scenario', fontsize=11)
ax.legend(fontsize=9)

# ---- Right: financial cost breakdown ----
ax2 = axes[1]
fn_costs = [s['FN'] * COST_FN for s in scenarios.values()]
fp_costs = [s['FP'] * COST_FP for s in scenarios.values()]
total_costs = [fn + fp for fn, fp in zip(fn_costs, fp_costs)]

bars1 = ax2.bar(x - w/2, fn_costs, width=w, label=f'FN cost (${COST_FN} each)', color='tomato', alpha=0.85)
bars2 = ax2.bar(x + w/2, fp_costs, width=w, label=f'FP cost (${COST_FP} each)', color='steelblue', alpha=0.85)

# Annotate total cost
for i, total in enumerate(total_costs):
    ax2.text(i, max(fn_costs[i], fp_costs[i]) + 200,
             f'Total: ${total:,.0f}', ha='center', va='bottom', fontsize=9,
             fontweight='bold', color='black')

ax2.set_xticks(x)
ax2.set_xticklabels(scenarios.keys(), rotation=10, ha='right', fontsize=9)
ax2.set_ylabel('Financial Cost (USD)')
ax2.set_title(f'Financial Cost per Scenario\n(FN=${COST_FN}/case, FP=${COST_FP}/case)', fontsize=11)
ax2.legend(fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

plt.suptitle('Business Impact of Threshold Choice in Fraud Detection',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Assumed: FN costs ${COST_FN}/fraud missed | FP costs ${COST_FP}/false alert")
for name, tc in zip(scenarios.keys(), total_costs):
    print(f"  {name:<35}: total cost = ${tc:,.0f}")

In [ ]:
# ── Cell 18: Metric summary table ─────────────────────────────────────────────

metric_summary = pd.DataFrame([
    {
        'Metric':      'Accuracy',
        'Formula':     '(TP+TN) / Total',
        'Business Question': 'Overall correctness',
        'Use For':     'Balanced classes only',
        'Fraud Use':   'AVOID — misleading at 3% fraud',
    },
    {
        'Metric':      'Precision',
        'Formula':     'TP / (TP + FP)',
        'Business Question': 'How trustworthy are alerts?',
        'Use For':     'Minimise false alarms',
        'Fraud Use':   'When investigation cost is high',
    },
    {
        'Metric':      'Recall',
        'Formula':     'TP / (TP + FN)',
        'Business Question': 'How much fraud do we catch?',
        'Use For':     'Minimise missed events',
        'Fraud Use':   'Primary metric for fraud teams',
    },
    {
        'Metric':      'F1 Score',
        'Formula':     '2PR / (P+R)',
        'Business Question': 'Balanced trade-off',
        'Use For':     'Equal FP/FN cost',
        'Fraud Use':   'When both costs roughly equal',
    },
    {
        'Metric':      'F2 Score (β=2)',
        'Formula':     '5PR / (4P+R)',
        'Business Question': 'Recall-weighted balance',
        'Use For':     'FN more costly than FP',
        'Fraud Use':   'FN costs 4x+ more than FP',
    },
    {
        'Metric':      'F0.5 Score (β=0.5)',
        'Formula':     '1.25PR / (0.25P+R)',
        'Business Question': 'Precision-weighted balance',
        'Use For':     'FP more costly than FN',
        'Fraud Use':   'VIP customers — FP very costly',
    },
    {
        'Metric':      'Specificity',
        'Formula':     'TN / (TN + FP)',
        'Business Question': 'How well do we pass legit?',
        'Use For':     'Customer experience focus',
        'Fraud Use':   'Track alongside recall',
    },
    {
        'Metric':      'Balanced Accuracy',
        'Formula':     '(Recall + Specificity) / 2',
        'Business Question': 'Balanced across classes',
        'Use For':     'Imbalanced class distribution',
        'Fraud Use':   'Alternative to F1 for imbalance',
    },
])

pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 140)
print("Classification Metrics Reference Table")
print("=" * 130)
print(metric_summary.to_string(index=False))

In [ ]:
# ── Cell 19: Complete dashboard — all metrics at a glance ─────────────────────

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# ---- Top-left: normalised confusion matrix for default threshold ----
ax = axes[0, 0]
cm_def = confusion_matrix(y_test, y_pred_05)
cm_def_norm = cm_def.astype(float) / cm_def.sum(axis=1, keepdims=True)
sns.heatmap(cm_def_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
            xticklabels=['Pred LEGIT', 'Pred FRAUD'],
            yticklabels=['Actual LEGIT', 'Actual FRAUD'],
            cbar_kws={'label': 'Rate'},
            linewidths=2, linecolor='white')
ax.set_title(f'Confusion Matrix (thr=0.50)\nTP={TP} FP={FP} TN={TN} FN={FN}', fontsize=10)

# ---- Top-right: metrics bar chart ----
ax2 = axes[0, 1]
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1', 'Specificity', 'Balanced\nAccuracy']
metric_vals   = [
    metrics_manual['Accuracy'],
    metrics_manual['Precision'],
    metrics_manual['Recall (TPR)'],
    metrics_manual['F1 Score'],
    metrics_manual['Specificity (TNR)'],
    metrics_manual['Balanced Accuracy']
]
colors_bar = ['grey', 'steelblue', 'tomato', 'seagreen', 'orchid', 'darkorange']
bars = ax2.barh(metric_names, metric_vals, color=colors_bar, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, metric_vals):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax2.set_xlim(0, 1.08)
ax2.axvline(0.5, color='black', linestyle=':', alpha=0.3)
ax2.set_xlabel('Score')
ax2.set_title('All Metrics at Default Threshold (0.50)\nAccuracy is misleadingly high!', fontsize=10)

# ---- Bottom-left: precision-recall tradeoff ----
ax3 = axes[1, 0]
ax3.plot(thresholds, results['precision'], color='steelblue', lw=2, label='Precision')
ax3.plot(thresholds, results['recall'],    color='tomato',    lw=2, label='Recall')
ax3.fill_between(thresholds, results['precision'], results['recall'],
                 alpha=0.12, color='purple', label='Gap (P-R tradeoff)')
for lbl, thr_val, col in [('F0.5', opt_f05, 'purple'), ('F1', opt_f1, 'seagreen'), ('F2', opt_f2, 'darkorange')]:
    ax3.axvline(thr_val, linestyle='--', alpha=0.7, color=col, label=f'Opt {lbl} thr={thr_val:.2f}')
ax3.set_xlabel('Threshold')
ax3.set_ylabel('Metric Value')
ax3.set_title('Precision-Recall Tradeoff vs Threshold', fontsize=10)
ax3.legend(fontsize=8)

# ---- Bottom-right: F-beta scores vs beta ----
ax4 = axes[1, 1]
betas   = [0.25, 0.5, 1.0, 2.0, 3.0]
f_maxes = [max((1+b**2)*p*r/(b**2*p+r+1e-9)
               for p, r in zip(results['precision'], results['recall']))
           for b in betas]
opt_rec = [results['recall'][np.argmax(
    [(1+b**2)*p*r/(b**2*p+r+1e-9) for p, r in zip(results['precision'], results['recall'])]
)] for b in betas]
opt_pre = [results['precision'][np.argmax(
    [(1+b**2)*p*r/(b**2*p+r+1e-9) for p, r in zip(results['precision'], results['recall'])]
)] for b in betas]

ax4.plot(betas, opt_rec, 'o-', color='tomato',    lw=2, label='Optimal Recall')
ax4.plot(betas, opt_pre, 's-', color='steelblue', lw=2, label='Optimal Precision')
ax4.plot(betas, f_maxes, '^-', color='seagreen',  lw=2, label='Max F-beta')
ax4.axvline(1, color='black', linestyle=':', alpha=0.5, label='β=1 (F1)')
ax4.set_xlabel('Beta (β)')
ax4.set_ylabel('Value at Optimal Threshold')
ax4.set_title('Effect of β on Optimal Precision/Recall\nLarger β → recall-focused threshold', fontsize=10)
ax4.legend(fontsize=8)

plt.suptitle('Classification Metrics — Complete Dashboard\nFraud Detection (3% fraud rate, n=5000)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 20: Why harmonic mean — arithmetic vs harmonic comparison ─────────────
# The harmonic mean penalises extreme imbalance between P and R

print("Harmonic Mean vs Arithmetic Mean: Why F1 Uses Harmonic\n")
print(f"{'Precision':>12} | {'Recall':>8} | {'Arithmetic (P+R)/2':>20} | {'Harmonic (F1)':>15} | {'Verdict'}")
print("-" * 80)

test_cases = [
    (0.80, 0.80, 'Balanced — both agree'),
    (0.95, 0.30, 'High P, low R — F1 penalises'),
    (0.30, 0.95, 'Low P, high R — F1 penalises'),
    (1.00, 0.01, 'Perfect precision, near-zero recall'),
    (0.50, 0.50, 'Moderate balanced'),
    (0.99, 0.99, 'Nearly perfect both'),
]

for prec, rec, desc in test_cases:
    arith  = (prec + rec) / 2
    harm   = 2 * prec * rec / (prec + rec + 1e-9)
    penalty = arith - harm
    print(f"{prec:>12.2f} | {rec:>8.2f} | {arith:>20.4f} | {harm:>15.4f} | "
          f"penalty={penalty:+.4f} | {desc}")

print("\nConclusion: When precision and recall are very different, harmonic mean")
print("is much lower than arithmetic mean — it PUNISHES extreme imbalance.")
print("This is why F1 is a better single metric than (P+R)/2 for fraud detection.")

In [ ]:
# ── Cell 21: Putting it all together — metric decision flowchart (code form) ───

def recommend_metric(class_imbalance: bool, fn_cost: float, fp_cost: float) -> str:
    """
    Given business context, recommend a primary evaluation metric.

    Parameters
    ----------
    class_imbalance : bool
        True if minority class < 10% of data.
    fn_cost : float
        Relative cost of a False Negative (missing fraud).
    fp_cost : float
        Relative cost of a False Positive (blocking innocent customer).

    Returns
    -------
    str : Recommended primary metric and reasoning.
    """
    ratio = fn_cost / (fp_cost + 1e-9)

    if not class_imbalance and abs(ratio - 1) < 0.2:
        return "Accuracy — balanced classes, equal costs. Clean signal."
    elif class_imbalance and abs(ratio - 1) < 0.5:
        return "Balanced Accuracy or F1 — imbalanced classes, roughly equal costs."
    elif ratio > 3:
        beta = round((ratio ** 0.5), 1)
        return f"F{beta} Score (recall-focused) — FN costs {ratio:.1f}x more than FP."
    elif ratio < 0.33:
        beta = round(1 / (ratio ** 0.5 + 1e-9), 1)
        return f"F{beta} Score (precision-focused) — FP costs {1/ratio:.1f}x more than FN."
    else:
        return "F1 Score — moderate asymmetry, balanced harmonic mean appropriate."


# Test with fraud detection scenarios
scenarios_meta = [
    (True,  500, 30,  "Standard fraud detection (FN=£500, FP=£30)"),
    (True,  500, 500, "High-stakes fraud AND high-stakes customer (equal costs)"),
    (False, 10,  10,  "Balanced spam filter (equal costs, balanced classes)"),
    (True,  100, 500, "VIP customer protection (FP very costly)"),
    (True,  2000, 10, "Medical diagnosis (FN catastrophic)"),
]

print("Metric Recommendation System")
print("=" * 80)
for imbal, fn_c, fp_c, desc in scenarios_meta:
    rec = recommend_metric(imbal, fn_c, fp_c)
    print(f"\n  Scenario: {desc}")
    print(f"  Recommendation: {rec}")

## Summary: The Metric Landscape

| Metric | Formula | Optimise When | Fraud Relevance |
|:---|:---|:---|:---|
| Accuracy | (TP+TN)/Total | Balanced classes, equal costs | AVOID — misleading at 3% fraud |
| Precision | TP/(TP+FP) | FP very costly | High-value customers; alert fatigue |
| Recall | TP/(TP+FN) | FN very costly | Primary fraud KPI — how much fraud stops |
| F1 | 2PR/(P+R) | FP = FN cost | Balanced view of model quality |
| F2 | 5PR/(4P+R) | FN >> FP cost | Fraud: missing fraud costs more than false alarms |
| F0.5 | 1.25PR/(0.25P+R) | FP >> FN cost | Spam: false alarms extremely annoying |
| Specificity | TN/(TN+FP) | Customer experience | Track alongside recall |
| Balanced Acc | (TPR+TNR)/2 | Imbalanced data | Alternative to F1 |

**The single most important takeaway:** The threshold is a business decision. Train your model once; tune the threshold for your business objective. Never use 0.5 by default for imbalanced problems.

---

## Self-Check Questions

---

**Question 1.** A fraud model has precision=0.95, recall=0.30. The bank says "we can't miss more than 20% of fraud." Is this model acceptable? What threshold change would you make?

<details>
<summary>Reveal Answer</summary>

**Not acceptable.** Recall = 0.30 means the model is missing 70% of fraud. The bank's constraint is recall ≥ 0.80 (miss no more than 20%).

To increase recall:
1. **Lower the classification threshold.** Currently the model is very conservative — it only flags transactions with very high fraud probability. By lowering the threshold (e.g., from 0.5 to 0.2), more transactions are flagged as fraud → fewer missed fraud cases → higher recall.

2. **Trade-off:** Lowering the threshold will reduce precision. At threshold=0.2 you will catch more fraud but also block more innocent customers. The bank must decide whether that is acceptable — this is the precision-recall tradeoff in action.

3. **Practical next step:** Plot recall vs threshold (as in Cell 11) and find the lowest threshold that achieves recall ≥ 0.80. Then check what precision you end up with at that threshold and assess whether the volume of false alarms is operationally manageable.

</details>

---

**Question 2.** Model A: precision=0.8, recall=0.8 (F1=0.8). Model B: precision=0.99, recall=0.31 (F1=0.47). When would you choose Model B over Model A?

<details>
<summary>Reveal Answer</summary>

Model B is nearly perfect at precision (99%) but only catches 31% of fraud — so it misses 69% of fraud cases. You would choose Model B when:

1. **The cost of a false alarm is catastrophically high.** For example, if blocking a transaction automatically freezes the card and notifies law enforcement, a single false positive on a VIP corporate account could cause severe relationship damage. In this case, you only want to flag fraud when you are nearly certain.

2. **Manual investigation capacity is very limited.** If your fraud operations team can only investigate 30 cases per day, Model B's high-precision alerts mean every investigated case is almost certainly real fraud — no wasted investigations.

3. **You have a complementary first-pass filter.** Model B could be the final decision layer, while a separate high-recall system (Model A) routes suspicious transactions to a human review queue. Model B then makes automatic blocks for only the clearest cases.

In most standard fraud detection contexts, Model A is better — catching 80% of fraud is far superior to catching 31%, and the false alarm rate is manageable.

</details>

---

**Question 3.** F-beta with β=2 weights recall more heavily. If you use this for fraud detection, what business assumption are you making about the relative costs of FP vs FN?

<details>
<summary>Reveal Answer</summary>

When β=2, the F-beta formula weights recall β² = 4 times more than precision. The formula is:

$$F_2 = \frac{5 \cdot \text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$$

Choosing F2 as your optimisation target implicitly assumes:

**The cost of a False Negative is approximately 4x the cost of a False Positive.**

In business terms for fraud detection:
- "Missing a fraudulent transaction ($500 average loss) is 4x worse than incorrectly blocking an innocent customer ($125 in service recovery, reputation, customer compensation)"
- The bank is saying: "We prefer to err on the side of caution — we would rather inconvenience a few innocent customers than let fraud slip through."

This is a reasonable assumption for many fraud contexts, especially where:
- Fraud losses are large and mostly unrecoverable
- False positive resolution (customer calls, compensation) has a known, manageable cost
- The bank has good customer service procedures to handle blocked transaction complaints quickly

**Key point:** β encodes a cost ratio. If your business team says "FN costs 9x more than FP", use β=3 (β² = 9). Always derive β from explicit cost estimates, not intuition.

</details>

---

**Question 4.** "Balanced accuracy" is useful for imbalanced datasets. Compute it: TP=45, FP=3, TN=4850, FN=2.

<details>
<summary>Reveal Answer</summary>

Balanced accuracy = (Sensitivity + Specificity) / 2

**Step 1 — Sensitivity (Recall):**
$$\text{Recall} = \frac{TP}{TP + FN} = \frac{45}{45 + 2} = \frac{45}{47} \approx 0.9574$$

**Step 2 — Specificity:**
$$\text{Specificity} = \frac{TN}{TN + FP} = \frac{4850}{4850 + 3} = \frac{4850}{4853} \approx 0.9994$$

**Step 3 — Balanced Accuracy:**
$$\text{Balanced Accuracy} = \frac{0.9574 + 0.9994}{2} = \frac{1.9568}{2} \approx 0.9784$$

**For comparison — ordinary accuracy:**
$$\text{Accuracy} = \frac{TP + TN}{\text{Total}} = \frac{45 + 4850}{4900} = \frac{4895}{4900} \approx 0.9990$$

**Interpretation:** Regular accuracy is 99.9% — almost entirely driven by how well we classify the 4853 legit transactions. Balanced accuracy (97.8%) better reflects model performance on BOTH classes, because it averages the per-class rates before they can be swamped by the majority class.

The model is doing well: it catches 45/47 = 95.7% of fraud and correctly passes 4850/4853 = 99.9% of legit transactions.

</details>